# Formatting reference databases

Formatting the EA riaz dataset from 22/10/2026 (final update) to fasta, BLAST, MAPseq and VSEARCH formats.

## Set-up

In [0]:
%sh
apt-get -q update

# BLAST+ installed
apt-get -q install -y ncbi-blast+

# vsearch install
apt-get -q install -y vsearch

# biopython install
pip install biopython

In [0]:
import pandas as pd
from Bio import SeqIO

In [0]:
# Load database
mfl_db = pd.read_csv(
    "/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom_Riaz/final_database_001.csv",
    encoding="cp1252"
)

# Load FASTA
fasta_df = pd.DataFrame(
    [(record.id, str(record.seq))
     for record in SeqIO.parse(
         "/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom_Riaz/final_database_001.fasta",
         "fasta"
     )],
    columns=["ID", "Sequence"]
)

In [0]:
print(fasta_df.head())

In [0]:
display(mfl_db.head())

## CSV to fasta (for BLAST)

In [0]:
# Write FASTA
with open("Data/Databases/EA_custom_Riaz/EA_fish_riaz_blast.fasta", "w") as f:
    for _, row in mfl_db.iterrows():
        f.write(f">{row['seq_id']} {row['Taxon_string']}\n")
        f.write(f"{row['Trimmed_sequence']}\n")

## Fasta to BLAST

In [0]:
%sh
# to make BLAST formatted database from curated db
makeblastdb -in /Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom_Riaz/EA_fish_riaz_blast.fasta -dbtype nucl -out Data/Databases/EA_custom_Riaz/EA_fish_riaz_db_FINAL/EA_fish_riaz_db_FINAL

## CSV to MAPseq

In [0]:
import csv
from collections import defaultdict

# ---- file paths ----
input_csv = "/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom_Riaz/final_database_001.csv"
base_dir = "/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom_Riaz/"

fasta_out = base_dir + "EA_riaz_mapseq.fasta"
tax_out = base_dir + "EA_riaz_mapseq.tax"
cluster_out = base_dir + "EA_riaz_mapseq.fasta.mscluster"

# ---- storage ----
records = []
species_to_ids = defaultdict(list)

def clean(x):
    if x is None:
        return "unknown"
    x = x.strip()
    return x.replace(" ", "_") if x and x != "NA" else "unknown"

# ---- load csv ----
with open(input_csv, newline='', encoding="cp1252") as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
             
        sequence = row["Trimmed_sequence"].strip().upper()
        seq_id = row["seq_id"]

        kingdom = clean(row.get("Kingdom"))
        phylum = clean(row.get("Phylum"))
        class_ = clean(row.get("Class"))
        order = clean(row.get("Order"))
        family = clean(row.get("Family"))
        genus = clean(row.get("Genus"))
        species = clean(row.get("Species"))

        # ensure species format
        if "_" not in species:
            species = f"{genus}_sp"

        records.append((seq_id, sequence, kingdom, phylum, class_, order, family, genus, species))
        species_to_ids[species].append(seq_id)

print(f"Loaded {len(records)} records")

# ---- write fasta ----
with open(fasta_out, "w") as f:
    for r in records:
        seq_id, sequence = r[0], r[1]
        f.write(f">{seq_id}\n{sequence}\n")

print("FASTA complete")

# ---- write taxa ----
with open(tax_out, "w") as f:

    # header          Kingdom   Phylum    Class     Order     Family    Genus     Species
    f.write("#cutoff: 0.00:0.08 0.70:0.35 0.70:0.35 0.70:0.35 0.80:0.25 0.92:0.08 0.95:0.05\n") # took from github values
    f.write("#name: MetaFish\n")
    f.write("#levels: Kingdom Phylum Class Order Family Genus Species\n")

    for r in records:
        seq_id, _, kingdom, phylum, class_, order, family, genus, species = r

        taxonomy = f"{kingdom};{phylum};{class_};{order};{family};{genus};{species}"

        f.write(f"{seq_id}\t{taxonomy}\n")

print("TAX file complete")

## CSV to DADA2 fasta

In [0]:
%skip
# no longer needed
# Load database
mfl_db = pd.read_csv(
    "/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom/20261008_EA_Reference_Library.csv",
    encoding="cp1252",
    low_memory=False
)

# Keep only 12S records
mfl_db = mfl_db[mfl_db["Gene"] == "12S"]

# Keep only records with taxonomy and sequence
mfl_db = mfl_db.dropna(
    subset=[
        "Sequence",
        "WoRMS_Kingdom",
        "WoRMS_Phylum",
        "WoRMS_Class",
        "WoRMS_Order",
        "WoRMS_Family",
        "WoRMS_Genus",
        "WoRMS_Species",
    ]
)

# Clean sequences
mfl_db["Sequence"] = (
    mfl_db["Sequence"]
    .str.upper()
    .str.replace(r"[^ACGT]", "", regex=True)
)

# Write FASTA
with open(
    "Data/Databases/EA_custom/20261008_EA_fish_riaz_taxonomy_AssignTaxonomy.fasta",
    "w"
) as f:
    for _, row in mfl_db.iterrows():

        header = (
            f">{row['WoRMS_Kingdom']};"
            f"{row['WoRMS_Phylum']};"
            f"{row['WoRMS_Class']};"
            f"{row['WoRMS_Order']};"
            f"{row['WoRMS_Family']};"
            f"{row['WoRMS_Genus']};"
            f"{row['WoRMS_Genus']} {row['WoRMS_Species']}"
        )

        f.write(header + "\n")
        f.write(row["Sequence"] + "\n")

## DADA2 to VSEARCH

In [0]:
%sh

# used dada2 fasta for this
fasta_file="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom_Riaz/final_database_001.fasta"
vsearch_db_output="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom_Riaz/vsearch_indexed_reference"

vsearch --makeudb_usearch "$fasta_file" --output "$vsearch_db_output"